# HSAMoE Non-Lesioned-Hemisphere Mechanism Probe

**STOPPED MECHANISM PROBE.** The non-lesioned-hemisphere arm reached 45.00% versus 49.17% for matched full montage and was worse for all three targets. Do not rerun or tune this exact channel mask; see `AGENTS.md` section 2e. Execution fails closed.

# 1. Setup

In [ ]:
raise RuntimeError('CLOSED HSAMoE hemisphere-mask probe: see AGENTS.md section 2e.')
import hashlib, json, platform, sys
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
WORKING_DIR = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / 'src' / 'liu2024').is_dir())
sys.path.insert(0, str(WORKING_DIR / 'src' / 'liu2024'))
from liu2024_candidate_runner import run_candidate_experiment
print(f'Python: {sys.version.split()[0]} | Platform: {platform.platform()} | Root: {WORKING_DIR}')

# 2. Configuration
## 2.1 CONFIG
Only the channel set differs between arms. Paralysis side is fixed clinical metadata and never estimated from EEG or outer-test labels.

In [ ]:
CONFIG = {
    # Paths / run identity
    'artifact_dir': str(WORKING_DIR / 'artifacts' / 'liu2024-hsamoe-nonlesioned-hemisphere'),
    'resume_run_dir': None,
    'source_extract_dir': str(WORKING_DIR / 'liu2024_data' / 'liu2024_figshare' / 'sourcedata'),
    'split_manifest_path': str(WORKING_DIR / 'artifacts' / 'liu2024-compact-mi-models' / '20260712_165746_790145_fd8ab986' / 'splits.json'),
    'experiment_name': 'hsamoe_nonlesioned_hemisphere_smoke_010307',
    'candidate_family': 'hemiparetic_side_shallow',
    'config_note': 'Prespecified HSAMoE mechanism gate: channels only; sub-01/03/07.',
    # Dataset / fixed clinical metadata
    'subjects_to_use': [1, 3, 7],
    'paralysis_side_by_subject': {'sub-01': 'right', 'sub-03': 'left', 'sub-07': 'right'},
    'target_sfreq': 128,
    'mi_window_seconds': 4.0,
    'average_reference': False,
    'bandpass_hz': [4.0, 40.0],
    'normalization_eps': 1e-6,
    # Matched model / training
    'model_kwargs': {},
    'n_epochs': 20,
    'batch_size': 8,
    'learning_rate': 3e-4,
    'weight_decay': 0.01,
    'gradient_clip_norm': 1.0,
    'model_seeds': [2026],
    'seed': 2026,
    # Spike gate; no inferential test for three patients
    'paired_inference_enabled': False,
    'gate_mean_delta_points': 1.0,
    'gate_max_collapse_increase_points': 5.0,
    'gate_max_class1_fraction_shift': 0.15,
}


## 2.2 Artifact Creation and Reproducibility
# 3. Load and Prepare Data
Complete trials are independently filtered and marker-aligned by `liu2024_prelocal_clean.py`.
# 4. Model
Matched ShallowFBCSPNet arms differ only in 29 versus 17 clinically fixed channels.
# 5. Training
All transforms and model fitting use only each outer fold's 32 training trials.
# 6. Results

In [ ]:
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + hashlib.md5(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:8]
ARTIFACT_DIR = Path(CONFIG['resume_run_dir']).resolve() if CONFIG.get('resume_run_dir') else Path(CONFIG['artifact_dir']) / RUN_ID
print(f'Run ID:     {RUN_ID}')
print(f'Artifacts:  {ARTIFACT_DIR}')
RUN_METADATA = run_candidate_experiment(CONFIG, ARTIFACT_DIR)
subject_metrics = pd.read_csv(ARTIFACT_DIR / 'subject_metrics.csv')
fold_results = json.loads((ARTIFACT_DIR / 'cv_results.json').read_text(encoding='utf-8'))
global_metrics = json.loads((ARTIFACT_DIR / 'global_metrics.json').read_text(encoding='utf-8'))
pivot = subject_metrics.pivot(index='subject_id', columns='method', values='balanced_accuracy')
deltas = pivot['nonlesioned_hemisphere'] - pivot['full_montage_control']
collapse = {}
for method in ['full_montage_control', 'nonlesioned_hemisphere']:
    rows = [row for row in fold_results if row['method'] == method]
    collapse[method] = float(np.mean([row['collapse_diagnostics']['single_class_prediction'] for row in rows]))
control_bias = global_metrics['methods']['full_montage_control']['predicted_class_1_fraction']
candidate_bias = global_metrics['methods']['nonlesioned_hemisphere']['predicted_class_1_fraction']
checks = {
    'mean_delta_at_least_one_point': float(deltas.mean() * 100) >= CONFIG['gate_mean_delta_points'],
    'not_worse_for_all_targets': bool((deltas >= 0).any()),
    'collapse_increase_within_limit': (collapse['nonlesioned_hemisphere'] - collapse['full_montage_control']) * 100 <= CONFIG['gate_max_collapse_increase_points'],
    'class_bias_shift_within_limit': abs(candidate_bias - control_bias) <= CONFIG['gate_max_class1_fraction_shift'],
}
gate = {
    'decision': 'EXPAND_FROZEN' if all(checks.values()) else 'STOP_CONFIGURATION',
    'checks': checks,
    'subject_deltas_points': {subject: float(value * 100) for subject, value in deltas.items()},
    'mean_delta_points': float(deltas.mean() * 100),
    'control_ba': float(pivot['full_montage_control'].mean()),
    'candidate_ba': float(pivot['nonlesioned_hemisphere'].mean()),
    'collapse_rate': collapse,
    'predicted_class_1_fraction': {'full_montage_control': control_bias, 'nonlesioned_hemisphere': candidate_bias},
    'inference_prohibited_for_three_patient_spike': True,
}
(ARTIFACT_DIR / 'gate_decision.json').write_text(json.dumps(gate, indent=2), encoding='utf-8')
print(json.dumps(gate, indent=2))
print(f'\nAll artifacts in: {ARTIFACT_DIR}')